In [ ]:
print("LAB 12 - Automação de Dados com Python e SQL")
print("Ambiente preparado com sucesso!")

LAB 12 - Automação de Dados com Python e SQL
Ambiente preparado com sucesso!


In [ ]:
import pandas as pd

dados = {
    "id_cliente": [1, 2, 3, 4, 5, 6],
    "nome": [
        "Ana Souza",
        "Carlos Lima",
        "Mariana Costa",
        "Ana Souza",
        "Pedro Santos",
        "Fernanda Alves"
    ],
    "email": [
        "ana@email.com",
        "carlos@email.com",
        None,
        "ana@email.com",
        "pedro@email.com",
        "fernanda@email.com"
    ],
    "telefone": [
        "11999990000",
        "11988887777",
        "21977776666",
        "11999990000",
        None,
        "21966665555"
    ]
}

df = pd.DataFrame(dados)

display(df)

,id_cliente,nome,email,telefone
0,1,Ana Souza,ana@email.com,11999990000
1,2,Carlos Lima,carlos@email.com,11988887777
2,3,Mariana Costa,None,21977776666
3,4,Ana Souza,ana@email.com,11999990000
4,5,Pedro Santos,pedro@email.com,None
5,6,Fernanda Alves,fernanda@email.com,21966665555


In [ ]:
print("DIAGNÓSTICO DA BASE DE CLIENTES")

print("\nQuantidade de registros:")
print(len(df))

print("\nCampos vazios por coluna:")
print(df.isnull().sum())

print("\nRegistros duplicados:")
print(df.duplicated(subset=["nome", "email"]).sum())

DIAGNÓSTICO DA BASE DE CLIENTES

Quantidade de registros:
6

Campos vazios por coluna:
id_cliente    0
nome          0
email         1
telefone      1
dtype: int64

Registros duplicados:
1


In [ ]:
# Criar uma cópia para preservar a base original
df_tratado = df.copy()

# Remover registros com e-mail duplicado
df_tratado = df_tratado.drop_duplicates(
    subset=["email"],
    keep="first"
)

# Identificar registros com informações pendentes
df_tratado["status_cadastro"] = "COMPLETO"

df_tratado.loc[
    df_tratado["email"].isnull(),
    "status_cadastro"
] = "REVISAR EMAIL"

df_tratado.loc[
    df_tratado["telefone"].isnull(),
    "status_cadastro"
] = "REVISAR TELEFONE"

display(df_tratado)

,id_cliente,nome,email,telefone,status_cadastro
0,1,Ana Souza,ana@email.com,11999990000,COMPLETO
1,2,Carlos Lima,carlos@email.com,11988887777,COMPLETO
2,3,Mariana Costa,None,21977776666,REVISAR EMAIL
4,5,Pedro Santos,pedro@email.com,None,REVISAR TELEFONE
5,6,Fernanda Alves,fernanda@email.com,21966665555,COMPLETO


In [ ]:
print("VALIDAÇÃO DA BASE TRATADA")

print("\nTotal de registros:")
print(len(df_tratado))

print("\nE-mails duplicados:")
print(df_tratado["email"].duplicated().sum())

print("\nCadastros por situação:")
print(df_tratado["status_cadastro"].value_counts())

print("\nCampos vazios:")
print(df_tratado.isnull().sum())

VALIDAÇÃO DA BASE TRATADA

Total de registros:
5

E-mails duplicados:
0

Cadastros por situação:
status_cadastro
COMPLETO            3
REVISAR EMAIL       1
REVISAR TELEFONE    1
Name: count, dtype: int64

Campos vazios:
id_cliente         0
nome               0
email              1
telefone           1
status_cadastro    0
dtype: int64


In [ ]:
import sqlite3

# Criar conexão com o banco de dados
conexao = sqlite3.connect("lab12_empresa.db")

# Salvar os dados tratados na tabela clientes
df_tratado.to_sql(
    "clientes",
    conexao,
    if_exists="replace",
    index=False
)

print("Banco de dados criado com sucesso!")
print("Tabela clientes criada e dados armazenados!")

Banco de dados criado com sucesso!
Tabela clientes criada e dados armazenados!


In [ ]:
consulta = """
SELECT *
FROM clientes;
"""

resultado = pd.read_sql_query(consulta, conexao)

display(resultado)

,id_cliente,nome,email,telefone,status_cadastro
0,1,Ana Souza,ana@email.com,11999990000,COMPLETO
1,2,Carlos Lima,carlos@email.com,11988887777,COMPLETO
2,3,Mariana Costa,None,21977776666,REVISAR EMAIL
3,5,Pedro Santos,pedro@email.com,None,REVISAR TELEFONE
4,6,Fernanda Alves,fernanda@email.com,21966665555,COMPLETO


In [9]:
consulta = """
SELECT
    status_cadastro,
    COUNT(*) AS quantidade
FROM clientes
GROUP BY status_cadastro;
"""

resultado = pd.read_sql_query(consulta, conexao)

display(resultado)

,status_cadastro,quantidade
0,COMPLETO,3
1,REVISAR EMAIL,1
2,REVISAR TELEFONE,1


In [10]:
pedidos = {
    "id_pedido": [101, 102, 103, 104, 105],
    "id_cliente": [1, 2, 1, 5, 6],
    "valor": [250.00, 480.00, 120.00, 350.00, 600.00],
    "status_pedido": [
        "ENTREGUE",
        "PENDENTE",
        "ENTREGUE",
        "EM PROCESSAMENTO",
        "ENTREGUE"
    ]
}

df_pedidos = pd.DataFrame(pedidos)

display(df_pedidos)

,id_pedido,id_cliente,valor,status_pedido
0,101,1,250.0,ENTREGUE
1,102,2,480.0,PENDENTE
2,103,1,120.0,ENTREGUE
3,104,5,350.0,EM PROCESSAMENTO
4,105,6,600.0,ENTREGUE


In [11]:
df_pedidos.to_sql(
    "pedidos",
    conexao,
    if_exists="replace",
    index=False
)

print("Tabela pedidos criada e dados armazenados com sucesso!")

Tabela pedidos criada e dados armazenados com sucesso!


In [12]:
consulta = """
SELECT
    pedidos.id_pedido,
    clientes.nome,
    pedidos.valor,
    pedidos.status_pedido
FROM pedidos
INNER JOIN clientes
    ON pedidos.id_cliente = clientes.id_cliente;
"""

resultado = pd.read_sql_query(consulta, conexao)

display(resultado)

,id_pedido,nome,valor,status_pedido
0,101,Ana Souza,250.0,ENTREGUE
1,102,Carlos Lima,480.0,PENDENTE
2,103,Ana Souza,120.0,ENTREGUE
3,104,Pedro Santos,350.0,EM PROCESSAMENTO
4,105,Fernanda Alves,600.0,ENTREGUE


In [13]:
consulta = """
SELECT
    clientes.nome,
    COUNT(pedidos.id_pedido) AS total_pedidos,
    SUM(pedidos.valor) AS valor_total
FROM clientes
INNER JOIN pedidos
    ON clientes.id_cliente = pedidos.id_cliente
GROUP BY clientes.nome
ORDER BY valor_total DESC;
"""

resultado = pd.read_sql_query(consulta, conexao)

display(resultado)

,nome,total_pedidos,valor_total
0,Fernanda Alves,1,600.0
1,Carlos Lima,1,480.0
2,Ana Souza,2,370.0
3,Pedro Santos,1,350.0


In [14]:
consulta = """
SELECT
    COUNT(*) AS quantidade_pedidos,
    SUM(valor) AS faturamento_total,
    AVG(valor) AS ticket_medio
FROM pedidos;
"""

resultado = pd.read_sql_query(consulta, conexao)

display(resultado)

,quantidade_pedidos,faturamento_total,ticket_medio
0,5,1800.0,360.0


In [15]:
# Consultar os dados consolidados
consulta = """
SELECT
    clientes.nome,
    COUNT(pedidos.id_pedido) AS total_pedidos,
    SUM(pedidos.valor) AS valor_total
FROM clientes
INNER JOIN pedidos
    ON clientes.id_cliente = pedidos.id_cliente
GROUP BY clientes.nome
ORDER BY valor_total DESC;
"""

relatorio = pd.read_sql_query(consulta, conexao)

# Exportar para Excel
relatorio.to_excel(
    "relatorio_vendas_lab12.xlsx",
    index=False
)

print("Relatório Excel gerado com sucesso!")
display(relatorio)

Relatório Excel gerado com sucesso!


,nome,total_pedidos,valor_total
0,Fernanda Alves,1,600.0
1,Carlos Lima,1,480.0
2,Ana Souza,2,370.0
3,Pedro Santos,1,350.0


In [16]:
import os

arquivo = "relatorio_vendas_lab12.xlsx"

if os.path.exists(arquivo):
    print("Arquivo localizado!")
    print("Tamanho:", os.path.getsize(arquivo), "bytes")
else:
    print("Arquivo não encontrado.")

Arquivo localizado!
Tamanho: 5061 bytes


In [18]:
from google.colab import files

files.download("relatorio_vendas_lab12.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>